# Preprocessing Validation (Sessions_In -> DuckDB)

This notebook preprocesses the minimal Sessions_In sample using the EDA CLI and stores the result in DuckDB.

Workflow:
1. Stage markdown files into a temporary vault-like Sessions folder.
2. Run `eda preprocess` to produce a parquet dataset.
3. Load parquet output into a DuckDB database table.
4. Query the table to validate frontmatter and description extraction.

In [7]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

import pandas as pd

from utils.duckdb_utils import open_duckdb, query_df
            


In [8]:
# Resolve workspace root robustly regardless of notebook cwd.
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
workspace = next((p for p in _candidates if (p / "src" / "eda").exists()), Path.cwd())
eda_src = workspace / "src" / "eda"
if str(eda_src) not in sys.path:
    sys.path.insert(0, str(eda_src))

print(f"workspace={workspace}")
print(f"eda_src={eda_src}")

workspace=c:\projects\fab\Fabcon2026
eda_src=c:\projects\fab\Fabcon2026\src\eda


In [9]:
source_dir = workspace / "Sessions"
staged_vault = workspace / "Processed" / "sessions_in_vault"
staged_sessions = staged_vault / "Sessions"

staged_sessions.mkdir(parents=True, exist_ok=True)
for md_file in source_dir.glob("*.md"):
    shutil.copy2(md_file, staged_sessions / md_file.name)

print(f"Staged {len(list(staged_sessions.glob('*.md')))} markdown file(s) in: {staged_sessions}")

Staged 245 markdown file(s) in: c:\projects\fab\Fabcon2026\Processed\sessions_in_vault\Sessions


In [10]:
output_base = workspace / "Processed" / "sessions_in_preprocessed_duckdb_source"
parquet_path = output_base.with_suffix(".parquet")

cmd = [
    sys.executable,
    "-m",
    "eda.main",
    "preprocess",
    "--vault",
    str(staged_vault),
    "--sessions-dir",
    "Sessions",
    "--no-workshops",
    "--no-tfidf",
    "--format",
    "parquet",
    "--output",
    str(output_base),
]

env = os.environ.copy()
env["PYTHONPATH"] = str((workspace / "src" / "eda").resolve()) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONIOENCODING"] = "utf-8"

result = subprocess.run(cmd, capture_output=True, text=False, env=env)
stdout = result.stdout.decode("utf-8", errors="replace")
stderr = result.stderr.decode("utf-8", errors="replace")
print(stdout)
if result.returncode != 0:
    print(stderr)
    raise RuntimeError(f"preprocess command failed with exit code {result.returncode}")

parquet_path

Loading sessions from: c:\projects\fab\Fabcon2026\Processed\sessions_in_vault
Loaded 245 sessions
         Dataset Overview         
┌────────────────────────┬───────┐
│ Metric                 │ Value │
├────────────────────────┼───────┤
│ Total sessions         │ 245   │
│   FABCON               │ 232   │
│   SQLCON               │ 13    │
│   Wednesday            │ 64    │
│   Thursday             │ 112   │
│   Friday               │ 69    │
│ Unique tracks          │ 21    │
│   Level 100            │ 31    │
│   Level 200            │ 103   │
│   Level 300            │ 101   │
│   Level 400            │ 6     │
│   Status: Not Reviewed │ 245   │
└────────────────────────┴───────┘
Saved → 
c:\projects\fab\Fabcon2026\Processed\sessions_in_preprocessed_duckdb_source.par
quet



WindowsPath('c:/projects/fab/Fabcon2026/Processed/sessions_in_preprocessed_duckdb_source.parquet')

In [11]:
duckdb_path = workspace / "Processed" / "sessions_in_preprocessed.duckdb"
with open_duckdb(duckdb_path, read_only=False) as con:
    con.execute("CREATE OR REPLACE TABLE sessions_in_preprocessed AS SELECT * FROM read_parquet(?)", [str(parquet_path)])
    count = con.execute("SELECT COUNT(*) FROM sessions_in_preprocessed").fetchone()[0]
print(f"DuckDB file: {duckdb_path}")
print(f"Rows loaded: {count}")
            


DuckDB file: c:\projects\fab\Fabcon2026\Processed\sessions_in_preprocessed.duckdb
Rows loaded: 245


In [12]:
query = '''
SELECT
  file,
  title,
  day,
  track,
  LEFT(description, 220) AS description_excerpt,
  LEFT(frontmatter_raw, 180) AS frontmatter_excerpt
FROM sessions_in_preprocessed
'''
preview = query_df(duckdb_path, query)
preview
            


,file,title,day,track,description_excerpt,frontmatter_excerpt
0,"90 Days, 1 Fabric Foundation- Delivering SQL M...","90 Days, 1 Fabric Foundation: Delivering SQL M...",Friday,Data Integration,"In this case study, Protective Industries tran...","title: ""90 Days, 1 Fabric Foundation: Deliveri..."
1,A data lake built for scale - deep dive into O...,A data lake built for scale: a deep dive into ...,Friday,OneLake,Join this session to learn how OneLake deliver...,"title: ""A data lake built for scale: a deep di..."
2,A Guide to Making the Most of your SQL Skills ...,A Guide to Making the Most of your SQL Skills ...,Thursday,SQL in Fabric,Don't start over. Learn how SQL professionals ...,"title: ""A Guide to Making the Most of your SQL..."
3,Accelerate Data Transformation with Dataflows ...,Accelerate Data Transformation with Dataflows ...,Thursday,Data Integration,## Key Takeaways\n\n-,"title: ""Accelerate Data Transformation with Da..."
4,Accelerate Your Move to Microsoft Fabric- Key ...,Accelerate Your Move to Microsoft Fabric: Key ...,Wednesday,Admin and Governance,Curious about how organizations cut months off...,"title: ""Accelerate Your Move to Microsoft Fabr..."
...,...,...,...,...,...,...
240,Whats new in Fabric capacities and capacity mo...,What's new in Fabric capacities and capacity m...,Thursday,Developer Experiences,## Key Takeaways\n\n-,"title: ""What's new in Fabric capacities and ca..."
241,Whats new in the SQL core engine - the 2026 ed...,What's new in the SQL core engine - the 2026 e...,Thursday,SQL Server,With all the news about developer and AI featu...,"title: ""What's new in the SQL core engine - th..."
242,When AI meets your database- Securing sensitiv...,When AI meets your database: Securing sensitiv...,Friday,Azure SQL,"As AI becomes embedded in every application, p...","title: ""When AI meets your database: Securing ..."
243,Where Data Meets Opportunity- Drive Business V...,Where Data Meets Opportunity: Drive Business V...,Thursday,Data Science,Join Esri to see how location intelligence can...,"title: ""Where Data Meets Opportunity: Drive Bu..."
